In [32]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 60.9 MB/s eta 0:00:00


In [33]:
import os, json, random, re, numpy as np, pandas as pd
from typing import List, Dict, Tuple
import warnings
warnings.filterwarnings('ignore')

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

try:
    import torch
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    torch.manual_seed(RANDOM_SEED)
except ImportError:
    device = 'cpu'

os.makedirs('artifacts', exist_ok=True)

In [ ]:
from urllib.parse import unquote

JSON_PATH = 'data/wikipedia_articles.json'

with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

knowledge_base = []
for i, item in enumerate(raw_data):
    url = item.get('url', f'unknown_{i}')
    text = item.get('text', '')
    # Декодируем URL и извлекаем заголовок
    title = unquote(url.split('/')[-1].replace('_', ' '))
    clean_text = re.sub(r'\s+', ' ', text).strip()
    knowledge_base.append({
        'id': f"doc_{i+1:03d}",
        'title': title,
        'url': url,
        'content': clean_text
    })

print(f"Документов загружено: {len(knowledge_base)}")
for doc in knowledge_base[:3]:
    print(f"{doc['title']}: {len(doc['content'])} симв.")

Документов загружено: 25
Новомарьясово: 1214 симв.
Сильный и слабый атеизм: 1203 симв.
Воротники: 602 симв.


In [35]:
def chunk_document(text: str, chunk_size: int = 400, overlap: int = 80) -> List[str]:
    chunks = []
    start = 0
    text = text.strip()
    while start < len(text):
        end = start + chunk_size
        if end < len(text):
            snippet = text[start:end]
            last_period = snippet.rfind('. ')
            if last_period > chunk_size // 2:
                end = start + last_period + 1
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
        if start >= len(text) or end >= len(text):
            break
    return chunks if chunks else [text]

CHUNK_SIZE = 400
CHUNK_OVERLAP = 80

all_chunks = []
chunk_metadata = []
for doc in knowledge_base:
    doc_chunks = chunk_document(doc['content'], CHUNK_SIZE, CHUNK_OVERLAP)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append(chunk)
        chunk_metadata.append({
            'doc_id': doc['id'],
            'doc_title': doc['title'],
            'doc_url': doc['url'],
            'chunk_idx': i,
            'chunk_text': chunk
        })

print(f"Всего чанков: {len(all_chunks)}")
print(f"Среднее на документ: {len(all_chunks)/len(knowledge_base):.1f}")

Всего чанков: 94
Среднее на документ: 3.8


In [36]:
try:
    from sentence_transformers import SentenceTransformer
    USE_ST = True
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
except ImportError:
    from sklearn.feature_extraction.text import TfidfVectorizer
    USE_ST = False
    vectorizer = TfidfVectorizer(max_features=384)

import faiss

if USE_ST:
    embeddings = embedding_model.encode(all_chunks, show_progress_bar=True, batch_size=32)
    faiss.normalize_L2(embeddings)
    embedding_dim = embeddings.shape[1]
else:
    embeddings = vectorizer.fit_transform(all_chunks).toarray()
    embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)
index.add(embeddings)
print(f"Индекс создан: {index.ntotal} векторов, размерность {embedding_dim}")

def search_faiss(query: str, top_k: int = 3) -> List[Tuple[int, float]]:
    if USE_ST:
        q_emb = embedding_model.encode([query])
        faiss.normalize_L2(q_emb)
    else:
        q_emb = vectorizer.transform([query]).toarray()
    distances, indices = index.search(q_emb, top_k)
    return [(idx, float(dist)) for idx, dist in zip(indices[0], distances[0])]

# Демонстрация
for q in ["Что такое искусственный интеллект?", "Машинное обучение определение"]:
    results = search_faiss(q, top_k=3)
    print(f"\nЗапрос: {q}")
    for rank, (idx, score) in enumerate(results, 1):
        print(f"  {rank}. {chunk_metadata[idx]['doc_title']} ({score:.3f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Индекс создан: 94 векторов, размерность 384

Запрос: Что такое искусственный интеллект?
  1. Иогансон, Андрей Борисович (0.582)
  2. Подорожник Корнута (0.571)
  3. Гринцер, Илья Моисеевич (0.530)

Запрос: Машинное обучение определение
  1. Подорожник Корнута (0.636)
  2. Иогансон, Андрей Борисович (0.592)
  3. Хороль (0.550)


In [37]:
control_queries = []
for i, doc in enumerate(knowledge_base[:10]):
    control_queries.append({
        "query": f"О чём статья {doc['title']}?",
        "expected_title": doc['title']
    })

TOP_K = 5
eval_results = []

for item in control_queries:
    query = item["query"]
    expected_title = item["expected_title"]
    results = search_faiss(query, top_k=TOP_K)
    retrieved_titles = [chunk_metadata[idx]['doc_title'] for idx, _ in results]
    hit = 1 if expected_title in retrieved_titles else 0
    try:
        rank = retrieved_titles.index(expected_title) + 1
    except ValueError:
        rank = -1
    eval_results.append({
        'query': query,
        'expected_source': expected_title,
        'retrieved_sources': '; '.join(retrieved_titles),
        'hit_at_k': hit,
        'rank_of_first_relevant': rank
    })

eval_df = pd.DataFrame(eval_results)
hit_at_k = eval_df['hit_at_k'].mean()
recall_at_k = hit_at_k
mrr = eval_df.apply(lambda r: 1/r['rank_of_first_relevant'] if r['rank_of_first_relevant'] > 0 else 0, axis=1).mean()

print(f"Hit@{TOP_K}: {hit_at_k:.3f}")
print(f"Recall@{TOP_K}: {recall_at_k:.3f}")
print(f"MRR@{TOP_K}: {mrr:.3f}")

eval_df.to_csv('homeworks/HW14/artifacts/retrieval_eval.csv', index=False)

Hit@5: 0.600
Recall@5: 0.600
MRR@5: 0.267


In [38]:
print("Эксперимент: chunk_size 300 vs 500")

def quick_eval(chunk_size: int) -> float:
    test_chunks = []
    for doc in knowledge_base[:15]:
        test_chunks.extend(chunk_document(doc['content'], chunk_size, CHUNK_OVERLAP))
    if USE_ST:
        embs = embedding_model.encode(test_chunks[:100], batch_size=32)
        faiss.normalize_L2(embs)
    else:
        embs = vectorizer.fit_transform(test_chunks[:100]).toarray()
    test_idx = faiss.IndexFlatIP(embedding_dim)
    test_idx.add(embs)
    hits = 0
    for item in control_queries[:5]:
        if USE_ST:
            q_emb = embedding_model.encode([item['query']])
            faiss.normalize_L2(q_emb)
        else:
            q_emb = vectorizer.transform([item['query']]).toarray()
        _, indices = test_idx.search(q_emb, TOP_K)
        retrieved = [test_chunks[i] if i < len(test_chunks) else "" for i in indices[0]]
        if any(item['expected_title'] in r for r in retrieved):
            hits += 1
    return hits / 5

for cs in [300, 500]:
    score = quick_eval(cs)
    print(f"chunk_size={cs}: Hit@{TOP_K} = {score:.3f}")

Эксперимент: chunk_size 300 vs 500
chunk_size=300: Hit@5 = 0.400
chunk_size=500: Hit@5 = 0.200


In [39]:
new_docs = [
    {'id': f"doc_{len(knowledge_base)+1:03d}", 'title': 'Этика ИИ', 'url': 'https://wiki.example/ethics',
     'content': 'Этика ИИ изучает моральные аспекты искусственного интеллекта: ответственность, предвзятость, приватность.'},
    {'id': f"doc_{len(knowledge_base)+2:03d}", 'title': 'Обучение с подкреплением', 'url': 'https://wiki.example/rl',
     'content': 'Обучение с подкреплением: агент учится через награды и штрафы. Алгоритмы: Q-learning, PPO, Policy Gradient.'},
    {'id': f"doc_{len(knowledge_base)+3:03d}", 'title': 'Трансформеры', 'url': 'https://wiki.example/transformers',
     'content': 'Трансформер — архитектура на основе внимания. Основа BERT, GPT. Применения: перевод, генерация, классификация.'}
]

test_queries = ["этика искусственного интеллекта", "обучение с подкреплением алгоритмы", "архитектура трансформеров"]

before_results = []
for q in test_queries:
    results = search_faiss(q, top_k=3)
    sources = [chunk_metadata[idx]['doc_title'] for idx, _ in results]
    before_results.append({'query': q, 'before_retrieved_sources': '; '.join(sources)})

updated_kb = knowledge_base + new_docs
updated_chunks = []
updated_meta = []
for doc in updated_kb:
    chunks = chunk_document(doc['content'], CHUNK_SIZE, CHUNK_OVERLAP)
    for i, ch in enumerate(chunks):
        updated_chunks.append(ch)
        updated_meta.append({
            'doc_id': doc['id'], 'doc_title': doc['title'], 'doc_url': doc['url'],
            'chunk_idx': i, 'chunk_text': ch
        })

if USE_ST:
    upd_embs = embedding_model.encode(updated_chunks, batch_size=32)
    faiss.normalize_L2(upd_embs)
else:
    upd_embs = vectorizer.fit_transform(updated_chunks).toarray()

upd_index = faiss.IndexFlatIP(embedding_dim)
upd_index.add(upd_embs)

def search_updated(query: str, top_k: int = 3):
    if USE_ST:
        q_emb = embedding_model.encode([query])
        faiss.normalize_L2(q_emb)
    else:
        q_emb = vectorizer.transform([query]).toarray()
    dists, inds = upd_index.search(q_emb, top_k)
    return [(idx, float(d)) for idx, d in zip(inds[0], dists[0])]

update_comparison = []
for i, q in enumerate(test_queries):
    results = search_updated(q, top_k=3)
    after_sources = '; '.join([updated_meta[idx]['doc_title'] for idx, _ in results])
    changed = before_results[i]['before_retrieved_sources'] != after_sources
    update_comparison.append({
        'query': q,
        'before_retrieved_sources': before_results[i]['before_retrieved_sources'],
        'after_retrieved_sources': after_sources,
        'changed': changed
    })
    print(f"Запрос: {q}\n  ДО: {before_results[i]['before_retrieved_sources']}\n  ПОСЛЕ: {after_sources}\n  Изменено: {changed}")

pd.DataFrame(update_comparison).to_csv('homeworks/HW14/artifacts/retrieval_before_after_update.csv', index=False)

Запрос: этика искусственного интеллекта
  ДО: Иогансон, Андрей Борисович; Подорожник Корнута; Гринцер, Илья Моисеевич
  ПОСЛЕ: Этика ИИ; Иогансон, Андрей Борисович; Подорожник Корнута
  Изменено: True
Запрос: обучение с подкреплением алгоритмы
  ДО: Иогансон, Андрей Борисович; Хороль; Подорожник Корнута
  ПОСЛЕ: Иогансон, Андрей Борисович; Хороль; Подорожник Корнута
  Изменено: False
Запрос: архитектура трансформеров
  ДО: Тетрагония; Иогансон, Андрей Борисович; Подорожник Корнута
  ПОСЛЕ: Трансформеры; Тетрагония; Иогансон, Андрей Борисович
  Изменено: True


In [47]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto" if torch.cuda.is_available() else None,
    trust_remote_code=True
)
model.eval()
print(f"Модель загружена на {next(model.parameters()).device}")

def generate_answer_llm(query: str, context: str, max_length: int = 256) -> str:
    prompt = f"Используй контекст для ответа на вопрос.\n\nКонтекст: {context}\n\nВопрос: {query}\n\nОтвет:"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=750,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    answer = full_text.split("Ответ:")[-1].strip()
    return answer

def mini_rag(query: str, top_k: int = 3) -> Dict:
    results = search_updated(query, top_k)
    chunks_text, sources = [], []
    for idx, _ in results:
        m = updated_meta[idx]
        chunks_text.append(m['chunk_text'])
        sources.append({'title': m['doc_title'], 'url': m['doc_url']})
    context = "\n\n".join(chunks_text)
    answer = generate_answer_llm(query, context)
    return {
        'question': query,
        'answer': answer,
        'retrieved_sources': sources
    }

rag_examples = []
rag_queries = [
    "Какое занятие было основным у жителей Новомарьясово?",
    "Кто впервые использовал термины сильный и слабый атеизм?",
    "Почему село Хороль так называется?",
    "Где проходила Потсдамская конференция?",
    "Какой статус у подорожника Корнута?"
]

for q in rag_queries:
    result = mini_rag(q, top_k=3)
    rag_examples.append({
        'question': result['question'],
        'answer': result['answer'],
        'retrieved_sources': '; '.join([s['title'] for s in result['retrieved_sources']])
    })
    print(f"\nВопрос: {q}")
    print(f"Ответ: {result['answer']}")
    print(f"Источники: {rag_examples[-1]['retrieved_sources']}")

pd.DataFrame(rag_examples).to_csv('homeworks/HW14/artifacts/rag_examples.csv', index=False)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Модель загружена на cuda:0

Вопрос: Какое занятие было основным у жителей Новомарьясово?
Ответ: В Новомарьясово было основное занятие - сельское хозяйство. В 1930-х годах в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1931 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1932 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1933 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1934 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1935 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1936 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1937 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1938 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1939 году в Новомарьясово было 1000 домов сельскохозяйственного хозяйства. В 1940 году в Новомарьясово было 1000 домов сельскохозяйственного хозяй